# Reproducing Kyzar et al. (2024) from dandiset 000469

Paper: Kyzar, Kamiński, Brzezicka, Reed, Chung, Mamelak & Rutishauser (2024), "Dataset of
human-single neuron activity during a Sternberg working memory task", Sci Data 11, 89,
doi:10.1038/s41597-024-02943-8.

Dataset: [DANDI:000469](https://dandiarchive.org/dandiset/000469/0.240123.1806), version
0.240123.1806 (doi:10.48324/dandi.000469/0.240123.1806). The paper cites the dandiset without a
version (ref. 46, "Kyzar M, 2023, DANDI Archive"), so the most recent published version is
pinned. The four published versions (0.231213.2047, 0.240122.1800, 0.240122.1812, 0.240123.1806)
and the draft list the same 41 assets with the same asset identifiers and sizes, so the choice
does not change the data read; the cell below checks this against the archive.

Text the claims file was written from: the version of record (Europe PMC, PMC10796636), with
Methods, Technical Validation, legends and Tables 1 to 3; the paper has no supplement. The
claims file also read the paper's code, github.com/rutishauserlab/workingmem-release-NWB at
commit f8594e0. Where a method detail below is taken from that code, the helper module
`sternberg.py` cites the file.

| Claim | Tested here |
|---|---|
| subjects-sessions | tested |
| sternberg-unit-yield | tested |
| screening-unit-yield | tested |
| behavior-accuracy | tested |
| behavior-rt-load | tested |
| behavior-accuracy-load | tested |
| isi-violations | tested |
| firing-rate | tested |
| projection-distance | tested |
| waveform-snr | tested |
| cv2 | tested |
| isolation-distance | tested |
| screening-concept-mtl | tested |
| screening-concept-mfc | tested |
| fig4-screening-examples | tested |
| sternberg-concept-mtl | tested |
| persistent-activity | tested |
| fig5-sternberg-examples | tested |
| maintenance-cells-mfc | tested |
| probe-cells-mfc | tested; the readings the criterion names are all run (see its section) |

In [ ]:
import numpy as np
import pandas as pd
import requests
from scipy import stats
import matplotlib.pyplot as plt

import sternberg as sb

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

In [ ]:
# Versions of the dandiset: the assets of each published version, compared with the pinned one.
api = f"https://api.dandiarchive.org/api/dandisets/{sb.DANDISET_ID}/versions/"
versions = [v["version"] for v in requests.get(api, timeout=60).json()["results"]]
def assets(version):
    r = requests.get(f"{api}{version}/assets/", params={"page_size": 200}, timeout=60).json()["results"]
    return sorted((a["path"], a["asset_id"], a["size"]) for a in r)
pinned = assets(sb.DANDISET_VERSION)
for v in versions:
    print(f"{v:>14}: {len(assets(v))} assets, identical to {sb.DANDISET_VERSION}: {assets(v) == pinned}")

In [ ]:
sessions = sb.load_all()
sc = [s for s in sessions if s["session"] == 1]
st = [s for s in sessions if s["session"] == 2]
units_sc, units_st = sb.units_table(sc), sb.units_table(st)
print(len(sessions), "files;", len(sc), "screening,", len(st), "Sternberg")

## Data check

What was loaded, the ranges and units of every variable used, and the checks of the description
against the data.

In [ ]:
# Files, subjects and sessions
files = pd.DataFrame([{"subject": s["subject"], "task": {1: "screening", 2: "Sternberg"}[s["session"]],
                       "identifier": s["identifier"], "age": s["age"], "sex": s["sex"],
                       "year": s["session_start_time"][:4], "units": len(s["units"]),
                       "trials": len(s["trials"]), "images": len(set(s["stim_ids"]))}
                      for s in sessions]).sort_values(["subject", "task"])
print(files.to_string(index=False))
native = files.assign(native=files.identifier.str.split("_", n=2).str[2])
print("\nsubject IDs:", files.subject.nunique(), "| native IDs:", native.native.nunique(),
      "| native IDs without the '_2' suffix:", native.native.str.replace("_2", "").nunique())
ages = files.drop_duplicates("subject").age.str.strip("PY").astype(int)
print("ages in the files:", ages.min(), "to", ages.max(), "years")

# Units: ids are 0..n-1 in every file (the code's 1-based unit index is id + 1)
print("unit ids 0..n-1 in every file:",
      all((s["units"].unit_id.to_numpy() == np.arange(len(s["units"]))).all() for s in sessions))
print("\nelectrode locations:", sorted(set(units_st.location) | set(units_sc.location)))
print("units by area and task:")
print(pd.DataFrame({"screening": units_sc.area.value_counts(), "Sternberg": units_st.area.value_counts()}))

# Spike times
for name, u in (("screening", units_sc), ("Sternberg", units_st)):
    allspk = np.concatenate(u.spike_times.to_list())
    unsorted = [(r.subject, r.unit_id, int(np.sum(np.diff(r.spike_times) < 0)),
                 float(np.diff(r.spike_times).min())) for r in u.itertuples() if np.any(np.diff(r.spike_times) < 0)]
    print(f"{name}: {len(allspk)} spikes, times {allspk.min():.1f} to {allspk.max():.1f} s; "
          f"spikes per unit {u.spike_times.map(len).min()} to {u.spike_times.map(len).max()}; "
          f"units with unsorted spike times (subject, unit, decreasing steps, most negative step s): {unsorted}")

# Identical spike trains deposited twice (first 20 spike times of every unit, all files)
seen = {}
for s in sessions:
    for r in s["units"].itertuples():
        seen.setdefault(tuple(np.round(np.sort(r.spike_times)[:20], 6)), []).append((s["path"], r.unit_id))
print("spike trains found in more than one place:", [v for v in seen.values() if len(v) > 1])

# Trials: Sternberg
t_all = pd.concat([s["trials"].assign(subject=s["subject"]) for s in st], ignore_index=True)
print("\nSternberg trials:", len(t_all), "| per session:", sorted(t_all.groupby("subject").size().unique()))
print("loads:", t_all.loads.value_counts().sort_index().to_dict(),
      "| Enc1 image IDs:", sorted(t_all.loadsEnc1_PicIDs.unique()),
      "| Enc2:", sorted(t_all.loadsEnc2_PicIDs.unique()), "| Enc3:", sorted(t_all.loadsEnc3_PicIDs.unique()),
      "| probe:", sorted(t_all.loadsProbe_PicIDs.unique()))
print("response_accuracy values:", sorted(t_all.response_accuracy.unique()),
      "| probe_in_out values:", sorted(t_all.probe_in_out.unique()))
print("probe_in_out agrees with the probe being among the encoded images:",
      ((t_all.loadsProbe_PicIDs == t_all.loadsEnc1_PicIDs) | (t_all.loadsProbe_PicIDs == t_all.loadsEnc2_PicIDs)
       | (t_all.loadsProbe_PicIDs == t_all.loadsEnc3_PicIDs)).astype(int).eq(t_all.probe_in_out).mean())
print("Enc2/Enc3 IDs are 0 exactly when the load leaves them out:",
      ((t_all.loadsEnc2_PicIDs == 0) == (t_all.loads < 2)).all(), ((t_all.loadsEnc3_PicIDs == 0) == (t_all.loads < 3)).all())
dur = pd.DataFrame({
    "fixation to Enc1 (s)": t_all.timestamps_Encoding1 - t_all.timestamps_FixationCross,
    "Enc1 shown (s)": t_all.timestamps_Encoding1_end - t_all.timestamps_Encoding1,
    "maintenance to probe (s)": t_all.timestamps_Probe - t_all.timestamps_Maintenance,
    "reaction time (s)": t_all.timestamps_Response - t_all.timestamps_Probe})
print(dur.describe(percentiles=[0.01, 0.5, 0.99]).round(3).to_string())
print("start_time equals timestamps_FixationCross:", (t_all.start_time == t_all.timestamps_FixationCross).all(),
      "| stop_time equals timestamps_Response:", (t_all.stop_time == t_all.timestamps_Response).all())
for k in (2, 3):
    on = t_all[f"timestamps_Encoding{k}"]
    print(f"Enc{k} timestamp is 0 or NaN exactly when the load leaves it out:",
          ((on.isna() | (on == 0)) == (t_all.loads < k)).all(), "| values when left out:",
          sorted(on[t_all.loads < k].fillna(-1).unique()), "| smallest when shown: %.1f s" % on[t_all.loads >= k].min())
odd = t_all[(dur["maintenance to probe (s)"] < 2) | (dur["fixation to Enc1 (s)"] > 2) | (dur["fixation to Enc1 (s)"] < 0.5)]
print("trials with maintenance under 2 s, or fixation under 0.5 s or over 2 s:")
print(odd[["subject", "loads", "timestamps_FixationCross", "timestamps_Encoding1", "timestamps_Encoding1_end",
           "timestamps_Maintenance", "timestamps_Probe"]].to_string())

# Trials: screening
for s in sc:
    assert np.allclose(s["trials"].start_time.to_numpy(), s["stim_times"])
counts = {s["subject"]: sorted(set(np.bincount(s["stim_ids"])[np.unique(s["stim_ids"])])) for s in sc}
print("\nscreening: start_time equals the StimulusPresentation timestamps in every file; presentations of each image:",
      sorted({c for v in counts.values() for c in v}), "| images per session:",
      sorted({len(set(s["stim_ids"])) for s in sc}))
dur_sc = np.concatenate([np.diff(s["trials"].start_time) for s in sc])
print("screening onset to next onset (s): median %.3f, 1st to 99th percentile %.3f to %.3f" %
      (np.median(dur_sc), *np.percentile(dur_sc, [1, 99])))

# Quality metrics stored per unit
print("\nstored quality metrics, Sternberg units:")
print(units_st[["isolation_distance", "proj_dist", "mean_snr", "peak_snr"]].describe().round(3).to_string())
print("NaN isolation distance:", units_st.isolation_distance.isna().sum(), "| zero projection distance:",
      (units_st.proj_dist == 0).sum())

The trial table timestamps are checked against the TTL stream in `acquisition/events` (codes from
Tables 2 and 3 of the paper). The spikes are checked against the trial clock with a contrast
between conditions: the fraction of units whose response 200 to 1000 ms after image onset differs
between images, with the onsets shifted by -2 to +2 s. On the right clock it peaks at 0 s; at
±1 s the window falls on the next or previous image (screening, and Encoding 2 in Sternberg).

In [ ]:
ttl = pd.DataFrame([dict(file=s["path"].split("/")[1], **sb.ttl_mismatch(s)) for s in sessions]).set_index("file")
print("trials whose timestamp has no TTL of the expected code within 1 ms, summed over files:")
print(ttl.sum().to_string())
shifts = (-2.0, -1.0, 0.0, 1.0, 2.0)
sel = pd.DataFrame([{"file": s["path"].split("/")[1], "task": s["session"], "units": len(s["units"]),
                     **sb.selectivity_by_shift(s, shifts)} for s in sessions]).set_index("file")
for task in (1, 2):
    sub = sel[sel.task == task]
    w = sub[list(shifts)].mul(sub.units, axis=0).sum() / sub.units.sum()
    print(f"\n{'screening' if task == 1 else 'Sternberg'}: fraction of selective units by shift (s), pooled:",
          w.round(3).to_dict())
not_peak = sel[sel[list(shifts)].idxmax(axis=1) != 0.0]
print("\nfiles where shift 0 is not the highest (clock not established by this check):")
print(not_peak.round(3).to_string())

## Unit and session counts

### subjects-sessions

> The dataset holds 41 recording sessions from 21 patients, every patient with a Sternberg
> session and all but one (P47HMH) with a screening session.

In [ ]:
by_subject = files.pivot_table(index="subject", columns="task", values="units", aggfunc="count").fillna(0).astype(int)
print("subjects:", files.subject.nunique(), "| sessions:", len(files), "| Sternberg sessions:",
      (files.task == "Sternberg").sum(), "| screening sessions:", (files.task == "screening").sum())
missing = by_subject[by_subject.screening == 0].index.tolist()
print("subjects without a screening session:", missing,
      files[files.subject.isin(missing)].identifier.tolist())
print("subjects 13 and 14:", files[files.subject.isin([13, 14])][["subject", "identifier", "age", "sex", "year"]]
      .drop_duplicates("subject").to_string(index=False))
print("ages:", ages.min(), "to", ages.max(), "(text 17 to 71; Table 1 17 to 70)")

**Result: reproduced.** The files hold 21 subject IDs, 41 sessions and 21 Sternberg sessions;
subject 19 (P47HMH) is the only one without a screening session. Subjects 13 and 14 carry the
native IDs P39CS and P39CS_2, with the same age and sex, so the 21 subjects may be 20 people.
Ages in the files run from 17 to 70, as in Table 1.

### sternberg-unit-yield

> In the Sternberg task 902 units were recorded, 190 in the hippocampus, 259 in the amygdala, 171
> in the dACC, 250 in the pre-SMA and 32 in the vmPFC.

In [ ]:
areas = ["Hipp", "Amg", "dACC", "pre-SMA", "vmPFC"]
print("Sternberg units:", len(units_st), units_st.area.value_counts().reindex(areas).to_dict())
print("MTL:", units_st.area.isin(sb.MTL).sum(), "| MFC:", units_st.area.isin(sb.MFC).sum())
print(pd.crosstab(units_st.subject, units_st.area).reindex(columns=areas, fill_value=0).T.to_string())

**Result: reproduced.** 902 Sternberg units: 190 hippocampus, 259 amygdala, 171 dACC, 250
pre-SMA and 32 vmPFC, every count equal to the paper's.

### screening-unit-yield

> In the screening task 907 units were recorded, 449 of them in the MTL.

In [ ]:
n_mtl_sc = units_sc.area.isin(sb.MTL).sum()
print("screening units:", len(units_sc), units_sc.area.value_counts().reindex(areas).to_dict())
print("MTL:", n_mtl_sc, "| MFC:", units_sc.area.isin(sb.MFC).sum())
for reading in (449, 421):
    print(f"MTL {n_mtl_sc} within 10% of {reading}: {abs(n_mtl_sc - reading) <= 0.1 * reading}")
print("Sternberg MTL for comparison:", units_st.area.isin(sb.MTL).sum())

**Result: reproduced.** 907 screening units, as stated. The MTL holds 421 of them (159
hippocampus, 262 amygdala), which is within 10% of 449 and equal to the 421 implied by the
screening percentages. 449 is the Sternberg MTL count, so the text appears to give the
Sternberg number for the screening task.

## Behaviour

### behavior-accuracy

> Subjects answered on average 88.73% of Sternberg trials correctly.

In [ ]:
acc = pd.Series({s["subject"]: 100 * s["trials"].response_accuracy.mean() for s in st}).sort_index()
print("percent correct per session:", acc.round(2).to_dict())
print("mean over sessions: %.2f%%, s.e.m. %.2f, range %.2f to %.2f" % (acc.mean(), acc.std() / np.sqrt(len(acc)), acc.min(), acc.max()))

**Result: reproduced.** Mean accuracy over the 21 sessions is 88.73%, the paper's value.

### behavior-rt-load

> Reaction times grew with memory load; a repeated-measures ANOVA of per-session median reaction
> times found a load effect.

In [ ]:
for label, correct in (("correct trials (main)", True), ("all trials (variant)", False)):
    m = sb.rt_medians(st, correct_only=correct)
    f, d1, d2, p = sb.rm_anova_load(m)
    print(f"{label}: mean of session medians by load (s) {m.mean(0).round(3).tolist()}; "
          f"F({d1},{d2}) = {f:.2f}, p = {p:.4f}; sessions with load 3 slower than load 1: {(m[:, 2] > m[:, 0]).sum()}/{len(m)}")
m = sb.rt_medians(st)
print("median RT range over sessions and loads: %.3f to %.3f s" % (m.min(), m.max()))
fig, ax = plt.subplots(figsize=(3.5, 3))
ax.plot([1, 2, 3], m.T, color="0.7", lw=0.8)
ax.plot([1, 2, 3], m.mean(0), color="C0", lw=2)
ax.set(xlabel="load", ylabel="median RT, correct trials (s)", xticks=[1, 2, 3])
plt.show()

**Result: reproduced.** On correct trials the mean of the session medians rises from 1.055 s
(load 1) to 1.130 and 1.198 s, and the load effect is F(2,40) = 7.92, p = 0.0013, the paper's
values. On all trials the effect is weaker (F(2,40) = 4.08, p = 0.024).

### behavior-accuracy-load

> Accuracy was lower at higher memory loads.

In [ ]:
acc_load = np.array([[100 * s["trials"][s["trials"].loads == L].response_accuracy.mean() for L in (1, 2, 3)] for s in st])
print("mean percent correct by load:", acc_load.mean(0).round(2).tolist())
print("per subject:")
print(pd.DataFrame(acc_load, index=[s["subject"] for s in st], columns=["load 1", "load 2", "load 3"]).round(1).sort_index().T.to_string())

**Result: reproduced.** Mean accuracy falls from 91.22% at load 1 to 88.62% at load 2 and
86.35% at load 3.

## Spike sorting quality (Sternberg units)

All six measures follow NWB_QA_graphs.m; spike times are used as stored, as the code does.

In [ ]:
isi = np.array([sb.isi_violation_pct(t) for t in units_st.spike_times])
rate = np.array([sb.mean_rate(t) for t in units_st.spike_times])
cv2s = np.array([sb.cv2(t) for t in units_st.spike_times])
print("units:", len(units_st))

### isi-violations

> Across Sternberg units, refractory period violations (ISIs shorter than 3 ms) were 0.41% of
> ISIs on average, with a standard deviation of 0.50%.

In [ ]:
print("all units (main reading): mean %.3f%%, s.d. %.3f%%, n = %d" % (isi.mean(), isi.std(ddof=1), len(isi)))
keep = isi < 4
print("units below 4%% (code's thresh_isi): mean %.3f%%, s.d. %.3f%%, n = %d; left out: %s" %
      (isi[keep].mean(), isi[keep].std(ddof=1), keep.sum(), np.round(isi[~keep], 2).tolist()))
print("range: %.3f to %.3f%%" % (isi.min(), isi.max()))

**Result: reproduced.** Over all 902 units the mean is 0.428% (within tolerance) and the s.d.
0.625% (outside 0.50 ± 0.05). With the code's filter, which leaves out one unit, the
values are 0.416% and 0.498%, both within tolerance; the claim holds on that reading. The
one unit left out has 11.74% of its ISIs below 3 ms.

### firing-rate

> Sternberg units fired at 2.55 Hz on average, ranging from 0.11 to 33.82 Hz.

In [ ]:
print("mean %.3f Hz, min %.3f Hz, max %.3f Hz, median %.3f Hz" % (rate.mean(), rate.min(), rate.max(), np.median(rate)))

**Result: reproduced.** Mean 2.546 Hz, range 0.108 to 33.824 Hz, against 2.55 and 0.11 to 33.82
Hz.

### projection-distance

> The pairwise projection distance between units isolated on the same wire was 13.77 plus or
> minus 7.32 s.d. of the signal.

In [ ]:
pdist = units_st.proj_dist.to_numpy()
nz = pdist[pdist != 0]
print("non-zero values: %d of %d; mean %.3f, s.d. %.3f, range %.2f to %.2f" % (len(nz), len(pdist), nz.mean(), nz.std(ddof=1), nz.min(), nz.max()))

**Result: reproduced.** Over the 734 non-zero stored values, 13.774 ± 7.325.

### waveform-snr

> The ratio between the s.d. of the noise and the peak amplitude of the mean waveform (an SNR)
> was 6.62 plus or minus 3.63 across Sternberg units.

In [ ]:
for col in ("peak_snr", "mean_snr"):
    v = units_st[col]
    print("%s: mean %.3f, s.d. %.3f, range %.2f to %.2f, n = %d" % (col, v.mean(), v.std(), v.min(), v.max(), v.notna().sum()))

**Result: reproduced.** The stored peak SNR (reading a) is 6.625 ± 3.632; the mean SNR (reading
b) is 1.647 ± 0.954 and does not match.

### cv2

> The modified coefficient of variation of the ISIs (CV2) was 0.94 plus or minus 0.14 across
> Sternberg units.

In [ ]:
print("CV2: mean %.3f, s.d. %.3f, range %.3f to %.3f, undefined %d" % (np.nanmean(cv2s), np.nanstd(cv2s, ddof=1), np.nanmin(cv2s), np.nanmax(cv2s), np.isnan(cv2s).sum()))

**Result: reproduced.** CV2 is 0.938 ± 0.138.

### isolation-distance

> The median isolation distance of Sternberg units was 31.14.

In [ ]:
iso = units_st.isolation_distance.dropna()
print("defined for %d of %d units; median %.3f, range %.2f to %.2f" % (len(iso), len(units_st), iso.median(), iso.min(), iso.max()))

**Result: reproduced.** Median 31.14 over the 676 units with a defined value.

## Selective cells

Tests follow the Methods ("Cell selection"): permutation tests with 2000 permutations. The code's
parametric tests are shown as a variant. Each claim's section has its own random generator.

### screening-concept-mtl

> In the screening task, 138 MTL units (32.78%) qualified as concept cells.

In [ ]:
rng_sc = np.random.default_rng(469_01)
conc_sc = sb.screening_concept(sc, rng_sc)
conc_sc["MTL"] = conc_sc.area.isin(sb.MTL)
conc_sc["MFC"] = conc_sc.area.isin(sb.MFC)
def summary(df, group):
    g = df[df[group]]
    return {"units": len(g), **{c: (int(g[c].sum()), round(100 * g[c].mean(), 2))
                                for c in ("concept_two", "concept_one", "parametric")}}
print("columns: (count, percent); concept_two = permutation tests, second test two-sided; "
      "concept_one = one-sided, larger; parametric = the code's anovan tests")
print("MTL:", summary(conc_sc, "MTL"))
print("MTL count as a percent of 449 (the text's MTL count):", {c: round(100 * conc_sc[conc_sc.MTL][c].sum() / 449, 2)
      for c in ("concept_two", "concept_one", "parametric")})
print("units passing the permuted ANOVA:", int((conc_sc[conc_sc.MTL].p_anova < 0.05).sum()), "MTL")
print("per subject (MTL units, concept_two):")
print(conc_sc[conc_sc.MTL].groupby("subject").agg(units=("concept_two", "size"), concept=("concept_two", "sum")).T.to_string())

**Result: reproduced.** Of 421 MTL screening units, 140 (33.25%) are concept cells under both
readings of the second test (two-sided and one-sided give the same cells); both values are
within tolerance of 138 and 32.78%. The code's parametric tests give 138 (32.78%), the paper's
numbers exactly.

### screening-concept-mfc

> In the screening task only 26 MFC units (5.35%) were concept cells, far fewer than in the MTL,
> so that concept cells are largely found in the MTL.

In [ ]:
print("MFC:", summary(conc_sc, "MFC"))
print("MTL percent (two-sided) %.2f%% against MFC %.2f%%" % (100 * conc_sc[conc_sc.MTL].concept_two.mean(),
                                                             100 * conc_sc[conc_sc.MFC].concept_two.mean()))
# The MFC count is near the upper bound of the tolerance (28.6 units, 5.885%): repeat with three other seeds.
mfc_sessions = [dict(s, units=s["units"][s["units"].area.isin(sb.MFC)]) for s in sc]
for seed in (469_11, 469_12, 469_13):
    rep = sb.screening_concept(mfc_sessions, np.random.default_rng(seed))
    print(f"seed {seed}: MFC concept cells two-sided {rep.concept_two.sum()} ({100 * rep.concept_two.mean():.2f}%), "
          f"one-sided {rep.concept_one.sum()} ({100 * rep.concept_one.mean():.2f}%)")
# Variant: units whose p lies between 0.02 and 0.10 for either test re-tested with 20000 permutations,
# to reduce the Monte Carlo error of 2000 permutations (same test, finer resolution).
refined = sb.screening_concept_refined(sc, conc_sc.drop(columns=["MTL", "MFC"]), np.random.default_rng(469_14))
for group, areas_ in (("MTL", sb.MTL), ("MFC", sb.MFC)):
    g = refined[refined.area.isin(areas_)]
    print(f"20000 permutations near 0.05 ({int(g.refined.sum())} {group} units re-tested): {group} concept cells "
          f"two-sided {int(g.concept_two.sum())} ({100 * g.concept_two.mean():.2f}%), "
          f"one-sided {int(g.concept_one.sum())} ({100 * g.concept_one.mean():.2f}%)")
print("per area (two-sided):", conc_sc.groupby("area").concept_two.agg(["sum", "size"]).T.to_dict())

**Result: reproduced.** Of 486 MFC screening units, 27 (5.56%) are concept cells under both
readings, within tolerance of 26 and 5.35%, and far below the MTL's 33.25%. The count sits near
the upper bound of the tolerance (28.6): with three other seeds it is 29, 28 and 27, and one of
them (29, 5.97%) is outside. Re-testing the 142 MFC units with a p-value between 0.02 and 0.10
with 20000 permutations, which reduces the Monte Carlo error of the same test, gives 26 (5.35%),
and 139 (33.02%) in the MTL. The code's parametric tests give 26 (5.35%), the paper's numbers.

### fig4-screening-examples

> Five example screening units (Fig. 4a to e) are concept cells.

In [ ]:
examples_sc = [(4, 16), (7, 24), (15, 3), (16, 22), (21, 15)]   # NWB_SB_plotCell_Screening.m, 1-based
rows = [conc_sc[(conc_sc.subject == s_) & (conc_sc.unit_id == u - 1)] for s_, u in examples_sc]
print(pd.concat(rows)[["subject", "unit_id", "area", "pref", "p_anova", "p_t_two", "p_t_one",
                       "concept_two", "concept_one", "parametric"]].to_string(index=False))

**Result: reproduced.** All five units pass the concept cell test under both readings (permuted
ANOVA p from 0.0005 to 0.028, second test p at most 0.001), and under the code's tests.

### sternberg-concept-mtl

> In the Sternberg task, 94 MTL units (20.94%) qualified as concept cells during the encoding
> period.

In [ ]:
rng_st = np.random.default_rng(469_02)
conc_st = sb.sternberg_concept(st, rng_st)
conc_st["MTL"] = conc_st.area.isin(sb.MTL)
conc_st["MFC"] = conc_st.area.isin(sb.MFC)
print("MTL:", summary(conc_st, "MTL"))
print("all areas:", summary(conc_st.assign(all=True), "all"))
print("per subject (MTL units, concept_two):")
print(conc_st[conc_st.MTL].groupby("subject").agg(units=("concept_two", "size"), concept=("concept_two", "sum")).T.to_string())# Repeat with other seeds: the one-sided count is near the upper bound of the tolerance (103.4 units).
for seed in (469_31, 469_32, 469_33):
    rep = sb.sternberg_concept(st, np.random.default_rng(seed))
    rep_mtl = rep[rep.area.isin(sb.MTL)]
    print(f"seed {seed}:", {k: (int(rep_mtl[k].sum()), round(100 * rep_mtl[k].mean(), 2),
                                abs(rep_mtl[k].sum() - 94) <= 9.4 and abs(100 * rep_mtl[k].mean() - 20.94) <= 2.094)
                            for k in ("concept_two", "concept_one")})


**Result: reproduced.** Of 449 MTL Sternberg units, 96 (21.38%) are concept cells with a
two-sided second test and 100 (22.27%) with a one-sided one; both are within tolerance of 94 and
20.94%. The code's parametric tests give 94 (20.94%), the paper's numbers. With three other seeds the counts are 95 to 97 (two-sided) and 99 to 100
(one-sided), all within tolerance.

### persistent-activity

> As a group, concept cells fired more during the maintenance period when their preferred image
> was held in memory than when it was not (4.76 against 2.67 Hz).

Readings, from the criterion: cells (a) MTL concept cells or (b) concept cells in any area;
preferred trials (i) the preferred image shown in any encoding period or (ii) as the Encoding 1
image. The concept cells are those of sternberg-concept-mtl, under both readings of its second
test. Mean and s.d. over cells of the maintenance rate (0 to 2500 ms), and a paired one-sided t-test.
The variant is the same comparison with the Encoding 1 window rates (200 to 1000 ms), as the
code computes it with the intended mask.

In [ ]:
expected = (4.76, 4.71, 2.67, 3.66)
tol = (0.476, 0.471, 0.267, 0.366)
rows = []
for test in ("concept_two", "concept_one", "parametric"):
    for cells in ("MTL", "all"):
        g = conc_st[conc_st[test] & (conc_st.MTL if cells == "MTL" else True)]
        for window in ("maint", "enc1"):
            for trials in ("any", "enc1"):
                a, b = g[f"{window}_pref_{trials}"], g[f"{window}_nonpref_{trials}"]
                vals = (a.mean(), a.std(), b.mean(), b.std())
                rows.append({"test": test, "cells": cells, "window": window, "pref trials": trials, "n": len(g),
                             "pref mean": vals[0], "pref sd": vals[1], "nonpref mean": vals[2], "nonpref sd": vals[3],
                             "p (paired, one-sided)": stats.ttest_rel(a, b, alternative="greater").pvalue,
                             "cells pref > nonpref": int((a > b).sum()),
                             "all four within tolerance": all(abs(v - e) <= t for v, e, t in zip(vals, expected, tol))})
pa = pd.DataFrame(rows)
print("window maint = 0-2500 ms after maintenance onset (the criterion); enc1 = 200-1000 ms after Encoding 1 onset (variant)")
print(pa.round({c: 3 for c in pa.columns if c.startswith(("pref ", "nonpref "))}).to_string(index=False, formatters={"p (paired, one-sided)": "{:.1e}".format}))

**Result: not reproduced.** In the maintenance window, under every cell and trial reading,
concept cells fire more on preferred trials (MTL cells, two-sided test, 2.57 against 2.12 Hz
with the preferred image in any encoding period, and 2.61 against 2.22 Hz with it as the
Encoding 1 image; all cells 2.94 against 2.57 and 2.99 against 2.65 Hz; every p below 1e-5),
but the preferred means are 2.57 to 3.13 Hz, far below the paper's 4.76 Hz, and no reading puts
the four values within tolerance. The paper's values come out exactly from the variant: all 119
concept cells of the code's parametric test, Encoding 1 window, preferred as the Encoding 1
image, give 4.76 ± 4.71 against 2.67 ± 3.66 Hz. With the permutation concept cells the same
variant gives 4.63 ± 4.64 against 2.59 ± 3.58 Hz (123 cells) and 4.75 ± 4.73 against 2.71 ±
3.65 Hz (128 cells), also within tolerance. The paper's numbers therefore appear to describe the
encoding response, not maintenance activity.

### fig5-sternberg-examples

> Two example amygdala units (Fig. 5a,b) are concept cells in the Sternberg task that also show
> persistent activity during maintenance.

In [ ]:
examples_st = [(7, 9), (16, 16)]   # NWB_SB_plotCell_Sternberg.m, 1-based
rows = [conc_st[(conc_st.subject == s_) & (conc_st.unit_id == u - 1)] for s_, u in examples_st]
print(pd.concat(rows)[["subject", "unit_id", "area", "pref", "p_anova", "p_t_two", "p_t_one", "concept_two",
                       "concept_one", "parametric", "maint_pref_any", "maint_nonpref_any"]].round(4).to_string(index=False))

**Result: reproduced.** Both units are amygdala concept cells under both readings (and the
code's tests), and both fire more during maintenance on trials that showed the preferred image
(5.05 against 1.43 Hz, and 5.73 against 4.17 Hz).

### maintenance-cells-mfc

> In the MFC, 144 units (31.79%) qualified as maintenance cells.

Readings, from the criterion: permutation by (a) label shuffling or (b) sign flipping; for
concept cells, non-preferred trials (i) Encoding 1 image not preferred or (ii) preferred image
not shown in any encoding period. Concept cells by the two-sided and one-sided tests of
sternberg-concept-mtl. Variant: the code's paired one-sided t-tests.

In [ ]:
rng_maint = np.random.default_rng(469_03)
maint = sb.sternberg_maintenance(st, conc_st, rng_maint)
maint = maint.merge(conc_st[["subject", "unit_id", "concept_two", "concept_one", "parametric"]], on=["subject", "unit_id"])
mfc = maint.area.isin(sb.MFC)
# The same permutations with ties counted as exceedances (>=), a variant
maint_ge = sb.sternberg_maintenance(st, conc_st, np.random.default_rng(469_03), ties=True)
maint_ge = maint_ge.merge(conc_st[["subject", "unit_id", "concept_two", "concept_one", "parametric"]], on=["subject", "unit_id"])
rows = []
for perm in ("shuffle", "flip", "t", "shuffle_ge", "flip_ge"):
    for nonpref in ("enc1", "any"):
        for test in ("concept_two", "concept_one", "parametric"):
            df, pm = (maint_ge, perm[:-3]) if perm.endswith("_ge") else (maint, perm)
            is_m = (df[f"p_{pm}"] < 0.05) & (~df[test] | (df[f"p2_{pm}_{nonpref}"] < 0.05))
            n = int(is_m[mfc].sum())
            rows.append({"permutation": perm, "non-preferred": nonpref, "concept test": test, "MFC maintenance": n,
                         "percent": round(100 * n / mfc.sum(), 2),
                         "within tolerance": abs(n - 144) <= 14.4 and abs(100 * n / mfc.sum() - 31.79) <= 3.179})
print("MFC units:", int(mfc.sum()), "(t = the code's paired t-tests, a variant; _ge = ties counted, a variant)")
print(pd.DataFrame(rows).to_string(index=False))
main = (maint.p_shuffle < 0.05) & (~maint.concept_two | (maint.p2_shuffle_enc1 < 0.05))
print("per area, label shuffling, Encoding 1 non-preferred:", maint[main].area.value_counts().to_dict())
print("per subject (MFC):", maint[mfc & main].groupby("subject").size().to_dict())

**Result: reproduced.** Of 453 MFC units, 139 to 144 (30.68% to 31.79%) are maintenance cells
under the four readings and every concept cell set, all within tolerance of 144 and 31.79%. The
code's paired t-tests give 144 (31.79%), the paper's numbers. Counting permuted
values equal to the observed one as exceedances (`>=`, a variant on the same permutations)
leaves label shuffling at 140 and 139 and lowers sign flipping to 142 and 141, all within
tolerance.

### probe-cells-mfc

> In the MFC, 122 units (26.93%) qualified as probe cells.

Readings, from the criterion: encoding rate (i) pooled over the encoding periods shown or (ii)
Encoding 1 only; permutation (a) label shuffling or (b) sign flipping. Variant: the code's paired
one-sided t-tests, with its Encoding 3 window at Encoding 2 onset.

In [ ]:
def probe_counts(df):
    out = {}
    for col in ("p_shuffle_pooled", "p_shuffle_enc1", "p_flip_pooled", "p_flip_enc1", "p_code"):
        n = int((df[col] < 0.05).sum())
        out[col] = (n, round(100 * n / len(df), 2), abs(n - 122) <= 12.2 and abs(100 * n / len(df) - 26.93) <= 2.693)
    return out
rng_probe = np.random.default_rng(469_04)
probe = sb.sternberg_probe(st, rng_probe, areas=sb.MFC)
print("MFC units:", len(probe), "| (count, percent, within tolerance) per reading:")
for k, v in probe_counts(probe).items():
    print(f"  {k}: {v}")
# The label-shuffling count is close to the lower bound of the tolerance (109.8 units): repeat with other seeds.
for seed in (469_21, 469_22, 469_23):
    rep = sb.sternberg_probe(st, np.random.default_rng(seed), areas=sb.MFC)
    print(f"seed {seed}:", {k: v for k, v in probe_counts(rep).items() if k != "p_code"})
# The same permutations as the main seed with ties counted as exceedances (>=), a variant
probe_ge = sb.sternberg_probe(st, np.random.default_rng(469_04), areas=sb.MFC, ties=True)
print("ties counted:", {k: v for k, v in probe_counts(probe_ge).items() if k != "p_code"})
print("per area (label shuffling, pooled):", probe[probe.p_shuffle_pooled < 0.05].area.value_counts().to_dict())

**Result: blocked (methods underspecified).** The readings disagree. With sign flipping, 118
(26.05%, pooled encoding) and 111 (24.50%, Encoding 1) units are probe cells, within tolerance.
With label shuffling, 109 (24.06%) and 104 (22.96%) are, outside it. With three other seeds,
label shuffling with pooled encoding gives 110 or 111, just inside the bound of 109.8, while
label shuffling with Encoding 1 gives 104 or 105 every time, outside it; sign flipping stays
within tolerance (110 to 124). Counting ties as exceedances (`>=`, same permutations as the main
seed) gives 108 and 103 for label shuffling, outside, and 118 and 111 for sign flipping, within
tolerance, so the readings still disagree. The code's paired
t-tests give 122 (26.93%), the paper's numbers. The Methods name "permutation t-tests" without
saying whether the trials are paired, and that choice decides the outcome.

## Summary

| Claim | Outcome |
|---|---|
| subjects-sessions | reproduced |
| sternberg-unit-yield | reproduced |
| screening-unit-yield | reproduced (MTL count is 421, not 449) |
| behavior-accuracy | reproduced |
| behavior-rt-load | reproduced |
| behavior-accuracy-load | reproduced |
| isi-violations | reproduced (on the code's filter) |
| firing-rate | reproduced |
| projection-distance | reproduced |
| waveform-snr | reproduced (peak SNR) |
| cv2 | reproduced |
| isolation-distance | reproduced |
| screening-concept-mtl | reproduced |
| screening-concept-mfc | reproduced |
| fig4-screening-examples | reproduced |
| sternberg-concept-mtl | reproduced |
| persistent-activity | not reproduced |
| fig5-sternberg-examples | reproduced |
| maintenance-cells-mfc | reproduced |
| probe-cells-mfc | blocked, methods underspecified |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the selection tests are the permutation tests the
Methods name, with 2000 permutations, where the paper's code runs parametric tests; the code's
tests are shown as variants and give the paper's counts exactly. The number of permutations and
the form of the permutation for paired rates are not given in the paper.

What was found about the dataset: it holds every session, unit and trial the paper describes,
and every number in the paper's quality and behaviour sections is recovered from it. The
paper's screening MTL count (449) is the Sternberg count; the files give 421. The paper's
persistent activity values are Encoding 1 window rates, not maintenance rates. Subjects 13 and 14
are P39CS and P39CS_2. Four units have spike times out of order. In one Sternberg trial (subject 9)
Encoding 1 lasts 2.5 s and the maintenance and probe onsets coincide, in another (subject 9)
the fixation cross and Encoding 1 onsets are 0.03 ms apart, so the baseline window falls on the
image, and in one (subject 21) the fixation cross lasts 14.07 s. These trials are kept, as the
code keeps them. Two column descriptions are wrong
(`waveforms_mean_proj_dist` is described as the isolation distance, `loadsEnc3_PicIDs` as Enc1).